# Gillon, Lecoq et al. (2023) and dandisets 000336 and 000037

Paper: Gillon CJ, Lecoq JA, et al. "Responses of pyramidal cell somata and apical dendrites in mouse
visual cortex over multiple days." Scientific Data 10, 287 (2023), doi:10.1038/s41597-023-02214-y.
The claims file (`claims/000336/10.1038-s41597-023-02214-y.yaml`) was written from the version of
record as served by Europe PMC (PMC10192216), with its tables and figure legends; Supplementary Table 1
was not available.

Target dataset: DANDI:000336, version 0.260325.1802, the only published version (the other is the
draft). It is the OpenScope "Dendritic Coupling" project. It names the paper among its related
resources, but the paper's data are on DANDI:000037 (OpenScope "Credit Assignment"), the dandiset the
paper's Data Records cite. The first claim tests whether 000336 holds any of the paper's sessions; all
other claims are marked `data: elsewhere` in the claims file and are tested on DANDI:000037.

Data used: DANDI:000037, version 0.230426.0054 (doi:10.48324/DANDI.000037/0.230426.0054). The paper
cites the dandiset without a version. The claims file names this version for every claim, and it is the
version published shortly before the paper (April 2023, paper published May 2023). The later version
0.240209.1623 adds only a `CHANGES` file (shown below), so the choice does not change the data read.
Only the basic files (`_behavior+ophys.nwb`) are read, streamed from the archive.

Outside input: the session table of the paper's code, `mouse_df.csv` of
github.com/colleenjg/OpenScope_CA_Analysis at commit 046e7fef5f452cdc0775fe54040b0f13aeae1e57, for the
session numbers (1 to 6 per mouse) and the QC status, which the files do not carry. The paper's Tables 1
and 2, copied from its text into the helper module, are used only as reference values.

| Claim | Tested here |
|---|---|
| dataset-holds-paper-sessions | yes, on 000336 |
| table1-subjects | yes |
| table2-sessions | yes |
| session-schedule | yes |
| imaging-planes | yes |
| roi-counts-per-session | yes |
| tracked-rois-per-plane | yes |
| fig8-example-matches | yes |
| fig8-ordering-variation | yes, in `roi_matching_orderings.ipynb` |
| stimulus-durations | yes |
| inconsistent-event-fraction | yes |
| fig5-stimulus-responses | yes |
| fig6-behaviour | yes |
| fig7a-snr-high-stable | yes |
| fig7b-signal-stable | yes |
| fig7c-correlations-similar | yes |
| high-correlation-pairs | yes |
| fig9-onset-offset-transients | yes |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import h5py, remfile, pynwb
import credassign as ca

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()

## The target, DANDI:000336

The archive's metadata, the file list, and one file opened and listed in full, including every column of
every table.

In [ ]:
m336 = ca.version_metadata(ca.TARGET_DANDISET, ca.TARGET_VERSION)
print(m336["name"])
print("published versions:", [v["version"] for v in ca.get_json(f"{ca.API}/dandisets/{ca.TARGET_DANDISET}/versions/")["results"]])
print("subjects (archive summary):", m336["assetsSummary"]["numberOfSubjects"], " files:", m336["assetsSummary"]["numberOfFiles"],
      " bytes:", m336["assetsSummary"]["numberOfBytes"])
for r in m336["relatedResource"]:
    print("related:", r["relation"], r.get("identifier", r["url"]), "|", r["name"])
a336 = ca.list_assets(ca.TARGET_DANDISET, ca.TARGET_VERSION)
a336["subject"] = a336.path.str.extract(r"sub-(\d+)/")[0]
a336["session"] = a336.path.str.extract(r"_ses-(\d+)")[0]
a336["kind"] = a336.path.str.extract(r"_([a-z+]+)\.nwb$")[0]
print(len(a336), "assets;", a336.subject.nunique(), "subjects;", a336.session.nunique(), "session IDs")
print(a336.groupby("kind").size().to_string())
print("subjects:", sorted(a336.subject.unique()))

In [ ]:
# One file of 000336, the smallest processed ophys file, listed in full.
small = a336[a336.kind == "ophys"].sort_values("size").iloc[0]
print(small.path, small["size"])
with pynwb.NWBHDF5IO(file=ca.open_h5(small.asset_id), load_namespaces=True) as io:
    nwb = io.read()
    print("session_start_time:", nwb.session_start_time, "| session_id:", nwb.session_id, "| identifier:", nwb.identifier)
    s = nwb.subject
    print("subject:", s.subject_id, s.genotype, s.sex, s.age, "|", s.description)
    for k, p in nwb.imaging_planes.items():
        print("imaging plane", k, "|", p.description, "|", p.location, "| rate", p.imaging_rate)
    print("acquisition:", {k: type(v).__name__ for k, v in nwb.acquisition.items()})
    for name, mod in nwb.processing.items():
        for dn, d in mod.data_interfaces.items():
            print("processing", name, dn, type(d).__name__)
            if hasattr(d, "colnames"):
                print("    columns:", list(d.colnames))
            for attr in ("roi_response_series", "time_series", "plane_segmentations"):
                for sn, ser in (getattr(d, attr, None) or {}).items():
                    print("   ", sn, getattr(getattr(ser, "data", None), "shape", None), list(getattr(ser, "colnames", [])))
    for k, t in nwb.intervals.items():
        print("intervals", k, len(t), "rows; columns:", list(t.colnames))

## The paper's data, DANDI:000037

The files list, the difference between its published versions, and one basic file listed in full.

In [ ]:
v37 = [v["version"] for v in ca.get_json(f"{ca.API}/dandisets/{ca.DANDISET}/versions/")["results"]]
print("versions of 000037:", v37)
a37 = ca.list_assets()
later = ca.list_assets(ca.DANDISET, "0.240209.1623")
both = a37.merge(later, on="path", how="outer", suffixes=("", "_later"), indicator=True)
print("only in the pinned version:", both.loc[both._merge == "left_only", "path"].tolist())
print("only in 0.240209.1623:", both.loc[both._merge == "right_only", "path"].tolist())
print("paths in both with a different size:", int((both.loc[both._merge == "both", "size"] != both.loc[both._merge == "both", "size_later"]).sum()))
a37["subject"] = a37.path.str.extract(r"sub-(\d+)/")[0]
a37["session"] = a37.path.str.extract(r"_ses-(\d+)_")[0]
a37["kind"] = a37.path.str.extract(r"_ses-\d+_(.*)\.nwb$")[0]
print(a37.groupby("kind").agg(n=("path", "size"), min_gb=("size", lambda x: x.min() / 1e9), max_gb=("size", lambda x: x.max() / 1e9)))
basic = a37[a37.kind == "behavior+ophys"].sort_values(["subject", "session"]).reset_index(drop=True)
print(len(basic), "basic files,", basic.session.nunique(), "sessions,", basic.subject.nunique(), "subjects")
print("each session has all three versions:", (a37.groupby("session").size() == 3).all())

In [ ]:
# One basic file of 000037 listed in full (subject 418779, session 777496949).
row = basic[basic.session == "777496949"].iloc[0]
with pynwb.NWBHDF5IO(file=ca.open_h5(row.asset_id), load_namespaces=True) as io:
    nwb = io.read()
    print("session_start_time:", nwb.session_start_time, "| session_id:", nwb.session_id)
    print("subject:", nwb.subject.subject_id, nwb.subject.genotype, nwb.subject.sex, nwb.subject.age)
    for k, p in nwb.imaging_planes.items():
        print("imaging plane", k, "| location", p.location, "| rate", p.imaging_rate, "| indicator", p.indicator)
    for name, mod in nwb.processing.items():
        for dn, d in mod.data_interfaces.items():
            for attr in ("roi_response_series", "time_series", "plane_segmentations", "images"):
                for sn, ser in (getattr(d, attr, None) or {}).items():
                    print(name, dn, sn, getattr(ser, "data", None).shape if hasattr(ser, "data") else "",
                          getattr(ser, "unit", ""), list(getattr(ser, "colnames", [])), getattr(ser, "description", ""))
    for col in nwb.trials.columns:
        print("trials column:", col.name, "|", col.description)
    tr = nwb.trials.to_dataframe()
    print(tr.groupby(["stimulus_type", "stimulus_template_name"]).size())
    print("acquisition:", list(nwb.acquisition), "| stimulus:", list(nwb.stimulus))

## Data check

All 50 basic files are opened for their metadata (subject, session, plane, stream lengths and the trial
table). The ROI traces of the 33 analysed sessions (sessions 1 to 3 that passed QC, by the paper's
code table) are read once, and every measure the claims need is computed as they load (helper,
`roi_measures`). This takes about eight minutes.

In [ ]:
mouse_df = ca.read_mouse_df()
mouse_df["sessid"] = mouse_df.sessid.astype(str)
mouse_df["mouseid"] = mouse_df.mouseid.astype(str)
table1, table2 = ca.paper_table1(), ca.paper_table2()
analysed = mouse_df[(mouse_df.pass_fail == "P") & mouse_df.sess_n.between(1, 3)]
ANALYSED = set(analysed.sessid)
SESS1 = set(analysed.loc[analysed.sess_n == 1, "sessid"])
print(len(ANALYSED), "analysed sessions from", analysed.mouseid.nunique(), "mice;", len(SESS1), "session-1 files")

meta, trials, roi, beh = {}, {}, {}, {}
for r in basic.itertuples():
    f = ca.open_h5(r.asset_id)
    meta[r.session] = ca.session_metadata(f)
    trials[r.session] = ca.trials_table(f)
    if r.session in ANALYSED:
        roi[r.session] = ca.roi_measures(f, trials[r.session], fig5=r.session in SESS1)
        beh[r.session] = ca.behaviour(f)
    f.close()
print(f"loaded in {(time.time() - T0) / 60:.1f} min")

In [ ]:
md_ = pd.DataFrame([{k: v for k, v in m.items() if k not in ("tracking_ids", "twop_timestamps")} for m in meta.values()])
md_["session"] = list(meta)
md_["depth"] = md_.location.str.extract(r"'depth': '(\d+)'")[0].astype(int)
md_["area"] = md_.location.str.extract(r"'area': '(\w+)'")[0]
md_["line"] = np.where(md_.genotype.str.startswith("Cux2"), "L2/3", np.where(md_.genotype.str.startswith("Rbp4"), "L5", "?"))
md_["plane"] = np.where(md_.depth.isin([175, 375]), "soma", "dend")
md_ = md_.merge(mouse_df[["sessid", "sess_n", "pass_fail", "full_timestamp", "date", "plane", "line", "nrois", "nrois_tracked"]]
                .rename(columns={"sessid": "session", "plane": "plane_code", "line": "line_code"}), on="session", how="left")
md_["analysed"] = md_.session.isin(ANALYSED)
print("files:", len(md_), "| subjects:", md_.subject_id.nunique(), "| sessions:", md_.session_id.nunique())
print("session_id equals the file name's session:", (md_.session_id == md_.session).all())
print("subject_id equals the file name's subject:", (md_.subject_id.values == basic.subject.values).all())
print(md_.groupby(["line", "plane"]).agg(mice=("subject_id", "nunique"), sessions=("session", "size"), analysed=("analysed", "sum")))
print("plane from depth agrees with the code table:", (md_.plane == md_.plane_code).all(),
      "| line from genotype agrees:", ((md_.line == "L2/3") == md_.line_code.str.startswith("L23")).all())
print("areas:", md_.area.unique(), "| imaging rates (metadata):", md_.imaging_rate.unique())
print("2p frame interval, s: median over files", md_.twop_median_dt.median(), "range", md_.twop_median_dt.min(), md_.twop_median_dt.max())
print("2p frames per session:", md_.twop_n.min(), "to", md_.twop_n.max(), "| dF/F shape frames equal timestamps:",
      all(m["dff_shape"][0] == m["twop_n"] for m in meta.values()), "| dF/F unit:", md_.dff_unit.unique())
print("running unit:", md_.run_unit.unique(), "| running sample interval (s):", md_.run_median_dt.min(), "to", md_.run_median_dt.max())
print("pupil unit:", md_.pupil_unit.unique(), "| pupil samples equal 2p frames:", (md_.pupil_n == md_.twop_n).all())
print("max projection shapes:", md_.max_projection_shape.astype(str).unique())
print("files whose pupil samples differ from their 2p frames:", md_.loc[md_.pupil_n != md_.twop_n, ["session", "pupil_n", "twop_n"]].to_dict("records"))
print("files without a tracking_id column:", int((~md_.has_tracking_column).sum()), "| their tracked count in Table 2:",
      sorted(set(table2.set_index("session_id").loc[md_.loc[~md_.has_tracking_column, "session"], "n_tracked"])))

In [ ]:
# Ranges of the variables used, analysed sessions.
r_ = pd.DataFrame({s: {"dff_min": v["dff_min"], "dff_max": v["dff_max"], "rois": v["n_rois"], "rois_nan_or_inf": v["n_bad"]}
                   for s, v in roi.items()}).T
print("dF/F over all ROIs and frames:", r_.dff_min.min(), "to", r_.dff_max.max(), "| sessions with NaN or Inf ROIs:", int((r_.rois_nan_or_inf > 0).sum()))
b_ = pd.DataFrame({s: {"run_min": np.min(v["run"]), "run_max": np.max(v["run"]), "run_dropped": v["run_dropped"],
                       "pup_min": np.min(v["pupil"]), "pup_max": np.max(v["pupil"]), "pup_nan": v["pupil_nan"],
                       "pup_dropped": v["pupil_dropped"]} for s, v in beh.items()}).T
print(b_.describe().loc[["min", "50%", "max"]])
# Duplicates: frame and ROI counts, and a fingerprint of the first 1000 frames of every ROI.
dup = md_.groupby(["twop_n", "n_rois"]).session.apply(list)
print("sessions sharing frame and ROI counts:", [v for v in dup if len(v) > 1])
fp = pd.Series({s: v["fingerprint"] for s, v in roi.items()})
print("analysed sessions sharing a trace fingerprint:", int(fp.duplicated(keep=False).sum()))

### Clocks

The trial table gives each stimulus row its 2p frames (`start_frame_twop`), and the 2p, pupil and
running series carry timestamps. The check below compares the table's frame numbers with the 2p
timestamps, and then uses a contrast: the population response at the stimulus onsets of the table,
against the same frames shifted by 5 to 20 s, where no onset is.

In [ ]:
lag = []
for s in sorted(ANALYSED):
    t, ts = trials[s], meta[s]["twop_timestamps"]
    fr = t.start_frame_twop.values.astype(int)
    d = ts[fr] - t.start_time.values          # 2p timestamp of each row's start frame minus the row's start time
    lag.append({"session": s, "median_s": np.median(d), "min_s": d.min(), "max_s": d.max()})
lag = pd.DataFrame(lag)
print("2p timestamp of each row's start frame - row start time, s: medians", lag.median_s.min().round(4), "to", lag.median_s.max().round(4),
      "| extremes", lag.min_s.min().round(4), "to", lag.max_s.max().round(4))
print("running timestamps span the stimuli in every session:",
      all((meta[s]["run_first"] <= trials[s].start_time.min() + 1) and (meta[s]["run_last"] >= trials[s].stop_time.max() - 1) for s in ANALYSED))
print("pupil timestamps equal the 2p timestamps in every analysed session:",
      all(meta[s]["pupil_n"] == meta[s]["twop_n"] and meta[s]["pupil_first"] == meta[s]["twop_first"] for s in ANALYSED))

In [ ]:
# Contrast: per session, the mean over ROIs of (mean dF/F 0-2 s after - 2 s before) at the table's
# onset frames, and at the same frames shifted by -20, -10, -5, +5, +10 and +20 s, in units of its SEM.
z = pd.DataFrame({s: {sh: v["onset_shift"][sh][0] / v["onset_shift"][sh][1] for sh in v["onset_shift"]} for s, v in roi.items()}).T
z.columns = [f"{c / ca.FPS:+.0f} s" for c in z.columns]
print(z.describe().loc[["min", "50%", "max"]].round(2).to_string())
true_col = "+0 s"
best = z[true_col].abs() >= z.drop(columns=true_col).abs().max(axis=1)
print("sessions where the table's onsets give the largest |mean/SEM| of all shifts:", int(best.sum()), "of", len(z))
print("sessions where they do not:")
print(z[~best].round(2).to_string())

The trial table's frame numbers agree with the 2p timestamps to within about one frame (printed above),
the pupil series shares the 2p timestamps, and the running series spans the stimuli. In 32 of 33
sessions the population response at the table's stimulus onsets is larger, in units of its SEM, than at
any shifted time; in the other, 758519303 (408021, session 1), the responses are weak at every shift
(1.6 SEM at the onsets, 2.5 at +5 s), so the check is not informative there. The running series could
not be checked by a contrast, as the claims use no running response.

## dataset-holds-paper-sessions

> The paper's data are the 50 sessions of Table 2, deposited on DANDI as dandiset 000037, with files
> named by subject and session ID. This claim checks whether dandiset 000336 holds any of those sessions.

Sessions are matched by subject and session ID from the file names of 000336 and the subject metadata
of its files, against the IDs of Table 2.

In [ ]:
paper_subjects, paper_sessions = set(table2.subject_id), set(table2.session_id)
print("Table 2:", len(paper_subjects), "subjects,", len(paper_sessions), "sessions")
print("000336 subjects in Table 2:", sorted(set(a336.subject) & paper_subjects))
print("000336 session IDs in Table 2:", sorted(set(a336.session.dropna()) & paper_sessions))
# subject metadata of one file per 000336 subject
subj = []
for s, g in a336[a336.kind == "ophys"].groupby("subject"):
    r = g.sort_values("size").iloc[0]
    with h5py.File(remfile.File(ca.s3_url(r.asset_id)), "r") as f:
        subj.append({"subject": s, "subject_id": f["general/subject/subject_id"][()].decode(),
                     "genotype": f["general/subject/genotype"][()].decode(),
                     "start": f["session_start_time"][()].decode()[:10]})
subj = pd.DataFrame(subj)
print(subj.to_string(index=False))
print("subject_id metadata in Table 2:", sorted(set(subj.subject_id) & paper_subjects))
print("000336 session start dates:", subj.start.min(), "to", subj.start.max(), "| Table 2 dates:", table2.date.min(), "to", table2.date.max())
print("000037 holds the 50 sessions of Table 2:", set(basic.session) == paper_sessions)

**Result: not reproduced.** None of the 16 subjects or 104 session IDs in the file names of DANDI:000336
is among the 13 subjects and 50 sessions of Table 2, nor is any subject ID in the metadata of the 14
subjects whose processed files were opened, and those sessions were recorded in 2022 and 2023, against
2018 and 2019 for the paper's. 000336 holds a different project. All 50 sessions of Table 2 are in
DANDI:000037, where the other claims are tested.

## table1-subjects

> 13 mice were imaged, 3 in L2/3-D, 3 in L2/3-S, 4 in L5-D and 3 in L5-S, 7 male and 6 female, each
> with at least three imaging sessions.

Line from the genotype (Cux2 is L2/3, Rbp4 is L5), plane from the imaging plane depth, sex from the
subject metadata, per subject, checked against Table 1. The date of birth is checked as the session date
minus the stated age.

In [ ]:
subj37 = md_.groupby("subject_id").agg(sex=("sex", "unique"), line=("line", "unique"), plane=("plane", "unique"),
                                       sessions=("session", "size"), genotype=("genotype", "first"))
for c in ["sex", "line", "plane"]:
    assert subj37[c].map(len).eq(1).all(), c
    subj37[c] = subj37[c].str[0]
# date of birth as start date (local, see session-schedule) minus age in days
md_["start_utc"] = pd.to_datetime(md_.session_start_time.str[:19])
md_["local_date"] = md_.start_utc.dt.tz_localize("UTC").dt.tz_convert("America/Los_Angeles").dt.strftime("%Y%m%d")
md_["age_days"] = md_.age.str.extract(r"P(\d+)D")[0].astype(int)
dob = (pd.to_datetime(md_.local_date) - pd.to_timedelta(md_.age_days, unit="D")).dt.strftime("%Y%m%d")
subj37["dob_from_age"] = dob.groupby(md_.subject_id).agg(lambda x: sorted(set(x)))
t1 = table1.set_index("subject_id")
t1["plane"] = t1.plane.map({"somata": "soma", "dendrites": "dend"})
cmp1 = subj37.merge(t1, left_index=True, right_index=True, suffixes=("", "_table1"))
print(cmp1[["genotype", "sex", "sex_table1", "line", "line_table1", "plane", "plane_table1", "sessions", "dob_from_age", "dob"]].to_string())
print("sex, line and plane agree with Table 1 for every subject:",
      all((cmp1[c] == cmp1[c + "_table1"]).all() for c in ["sex", "line", "plane"]))
grp = subj37.groupby(["line", "plane"]).size()
print(grp.to_string())
print("mice:", len(subj37), "| male:", (subj37.sex == "M").sum(), "| female:", (subj37.sex == "F").sum(),
      "| fewest sessions per mouse:", subj37.sessions.min())

**Result: reproduced.** The files hold 13 mice: 3 L2/3-D, 3 L2/3-S, 4 L5-D and 3 L5-S, 7 male and 6
female, each with at least 3 sessions. Sex, line and plane agree with Table 1 for every subject, and the
date of birth implied by each file's age agrees with Table 1.

## table2-sessions

> 50 imaging sessions were recorded from the 13 mice. Two mice have sessions that failed QC; the
> sessions 1 to 3 that passed QC are 33 files.

The files carry no QC status; it comes from the paper's code table (`mouse_df.csv`), checked here
against Table 2. Readings for the mice with failed sessions: (a) mice whose sessions 1 to 3 include a
failed session, (b) mice with any failed session.

In [ ]:
t2 = table2.set_index("session_id")
qc = mouse_df.set_index("sessid")
print("sessions in the files:", md_.session.nunique(), "| in Table 2:", len(t2), "| same set:", set(md_.session) == set(t2.index))
print("code table QC agrees with Table 2:", ((qc.loc[t2.index, "pass_fail"] == "P") == (t2.qc == "passed")).all())
failed = t2[t2.qc == "failed"]
print("failed sessions:", failed.groupby("subject_id").size().to_dict())
s123 = mouse_df[mouse_df.sess_n.between(1, 3)]
read_a = sorted(s123.loc[s123.pass_fail == "F", "mouseid"].unique())
read_b = sorted(failed.subject_id.unique())
print("(a) mice with a failed session among sessions 1 to 3:", len(read_a), read_a)
print("(b) mice with any failed session:", len(read_b), read_b)
print("sessions 1 to 3 that passed QC:", len(ANALYSED), "files, from", analysed.mouseid.nunique(), "mice")
print("sessions per mouse in the files:", md_.groupby("subject_id").size().to_dict())

**Result: reproduced.** The files hold 50 sessions, the set of Table 2. Under reading (a), 2 mice (411400
and 412933) have a failed session among sessions 1 to 3; under reading (b) the count is 3, with 440889,
whose sessions 5 and 6 failed. 33 sessions 1 to 3 passed QC. The QC status is not in the files; it comes
from the paper's code table, which agrees with Table 2.

## session-schedule

> After habituation each mouse had three 70-minute imaging sessions within three to six days, at most
> one per day.

The files' `session_start_time` is stored with a -08:00 offset, but its clock time equals the UTC start
time of the code table (`full_timestamp`), so the stored offset is wrong: the start times are UTC. Local
dates are taken by converting them from UTC to Pacific time, and compared with Table 2's dates. Readings:
which sessions, (a) sessions 1 to 3 as the code table numbers them, (b) the three tracked sessions (a
nonzero tracked count in the files); span, (i) days from first to third, (ii) calendar days inclusive;
duration, (x) first to last 2p frame, (y) first stimulus row start to last stop, grey screen included.

In [ ]:
md_["start_clock"] = md_.session_start_time.str[:19].str.replace("-", "").str.replace(":", "")
print("stored offsets:", md_.session_start_time.str[-6:].unique())
print("start clock time equals the code table's UTC timestamp in every file:", (md_.start_clock == md_.full_timestamp).all())
md_["utc_date"] = md_.start_utc.dt.strftime("%Y%m%d")
print("local dates equal Table 2's dates:", (md_.set_index("session").local_date == t2.date.reindex(md_.session).values).all(),
      "| UTC dates equal them:", (md_.set_index("session").utc_date == t2.date.reindex(md_.session).values).sum(), "of", len(md_))
md_["dur_x"] = (md_.twop_last - md_.twop_first) / 60
md_["dur_y"] = [(trials[s].stop_time.max() - trials[s].start_time.min()) / 60 for s in md_.session]
print("duration (x), min:", md_.dur_x.min().round(2), "to", md_.dur_x.max().round(2),
      "| (y):", md_.dur_y.min().round(2), "to", md_.dur_y.max().round(2))

def spans(sel, datecol):
    out = {}
    for mouse, g in sel.groupby("subject_id"):
        d = pd.to_datetime(g[datecol]).sort_values()
        out[mouse] = {"n": len(d), "distinct": d.nunique() == len(d), "days_i": (d.iloc[-1] - d.iloc[0]).days,
                      "days_ii": (d.iloc[-1] - d.iloc[0]).days + 1, "dates": " ".join(d.dt.strftime("%m%d"))}
    return pd.DataFrame(out).T

sel_a = md_[md_.sess_n.between(1, 3)]
sel_b = md_[md_.n_tracked > 0]
for name, sel in (("a", sel_a), ("b", sel_b)):
    for datecol in ("local_date", "utc_date"):
        sp = spans(sel, datecol)
        ok_i = sp.distinct & sp.days_i.between(3, 6) & (sp.n == 3)
        ok_ii = sp.distinct & sp.days_ii.between(3, 6) & (sp.n == 3)
        print(f"\nreading ({name}), {datecol}: mice {len(sp)}, three sessions each: {(sp.n == 3).all()}; "
              f"span (i) in 3-6 for {ok_i.sum()}, span (ii) in 3-6 for {ok_ii.sum()}")
        if datecol == "local_date":
            print(sp.to_string())
for name, sel in (("a", sel_a), ("b", sel_b)):
    print(f"durations under ({name}): (x) {sel.dur_x.min():.2f} to {sel.dur_x.max():.2f} min (median {sel.dur_x.median():.2f}), "
          f"(y) {sel.dur_y.min():.2f} to {sel.dur_y.max():.2f} min (median {sel.dur_y.median():.2f})")

**Result: reproduced.** Under readings (a) and (ii), sessions 1 to 3 of each of the 13 mice fall on
three distinct local dates within 3 to 6 calendar days, and every session lasts 70.2 to 70.5 minutes
under duration reading (x) and 70.1 to 70.4 under (y). Under span (i) (days from the first session to
the third) only 6 of 13 mice are within 3 to 6 days, as 7 had their sessions on three consecutive
days (2 days). With the tracked sessions (b), 411400 spans 7 calendar days. The local dates match Table 2
for all 50 sessions once the stored start times are read as UTC; the UTC dates match for 46.

## imaging-planes

> Somata were imaged at 175 um (L2/3) and 375 um (L5) and distal apical dendrites at 50 to 75 um (L2/3)
> and 20 um (L5), at 30 Hz with 512 x 512 pixels over a 400 um field of view.

In [ ]:
d = md_.set_index("session")
print("depth equals Table 2 in", (d.depth == t2.depth.reindex(d.index)).sum(), "of", len(d), "sessions")
mis = d[d.depth != t2.depth.reindex(d.index)]
print("sessions whose depth differs from Table 2:")
print(pd.DataFrame({"subject": mis.subject_id, "sess_n": mis.sess_n, "file_depth": mis.depth, "table2_depth": t2.depth.reindex(mis.index),
                    "code_table_depth": mouse_df.set_index("sessid").depth.reindex(mis.index)}).to_string())
print(md_.groupby(["line", "plane"]).depth.unique().to_string())
print("imaging rate metadata:", md_.imaging_rate.unique(), "Hz | from timestamps:",
      (1 / md_.twop_median_dt).min().round(3), "to", (1 / md_.twop_median_dt).max().round(3), "Hz")
print("max projection image sizes:", md_.max_projection_shape.astype(str).value_counts().to_dict())
print("field of view in um: not stated in the files (imaging plane grid spacing not set)")

**Result: partially reproduced.** The imaging rate is 30 Hz in the metadata and 30.08 to 30.15 Hz from
the timestamps, and every plane image is 512 x 512 pixels. The depth equals Table 2 in 47 of 50
sessions: the three sessions of 433414 (L2/3-D) are at 60 um in the files and at 75 um in Table 2 and
in the paper's code table. The 400 um field of view is not in the files.

## roi-counts-per-session

> 15 to 224 ROIs per session were retained in the somatic planes and 159 to 1636 in the dendritic
> planes.

ROI count per session from the files, against Table 2. Readings: (a) the 33 analysed sessions, (b)
every session that passed QC, (c) all 50 sessions. Variant: ROIs left after removing those with NaN or
Inf values (traces read for the 33 analysed sessions only).

In [ ]:
d = md_.set_index("session")
print("ROI count equals Table 2 in", (d.n_rois == t2.n_rois.reindex(d.index)).sum(), "of", len(d), "sessions")
passed = set(t2.index[t2.qc == "passed"])
for name, sel in (("a", d[d.index.isin(ANALYSED)]), ("b", d[d.index.isin(passed)]), ("c", d)):
    g = sel.groupby("plane").n_rois.agg(["min", "max", "size"])
    print(f"({name})", g.to_dict("index"))
clean = pd.Series({s: v["n_rois"] - v["n_bad"] for s, v in roi.items()})
print("variant, after NaN/Inf removal (a):", clean.groupby(d.loc[clean.index, "plane"]).agg(["min", "max"]).to_dict("index"),
      "| ROIs removed:", int(sum(v["n_bad"] for v in roi.values())))

**Result: reproduced.** Every session's ROI count equals Table 2. Under readings (a) and (b) the somatic
counts run from 15 to 224 and the dendritic from 159 to 1636, the paper's ranges exactly; under reading
(c) the largest somatic count is 227, inside the widened range. No ROI of the 33 analysed sessions has
NaN or Inf values, so the variant gives the same counts.

## tracked-rois-per-plane

> ROI matching across the three sessions gave 254 tracked ROIs in L2/3-D, 261 in L2/3-S, 516 in L5-D
> and 129 in L5-S, about 32 to 75% of the theoretical maximum.

Tracked ROIs are those with a tracking index (`tracking_id` not NaN). Per mouse, the tracked count must
be the same in its three tracked sessions, and the indices must run over the same set. Theoretical
maximum readings: (a) per plane, the sum over mice of the smallest ROI count among the three sessions;
(b) per plane, the sum of the mean count; (c) per mouse, as (a).

In [ ]:
trk = []
for s, m in meta.items():
    ids = m["tracking_ids"]
    trk.append({"session": s, "subject_id": m["subject_id"], "n_tracked": m["n_tracked"], "n_rois": m["n_rois"],
                "ids": tuple(sorted(ids[~np.isnan(ids)].astype(int)))})
trk = pd.DataFrame(trk).merge(md_[["session", "line", "plane"]], on="session")
trk["group"] = trk.line + "-" + trk.plane.map({"soma": "S", "dend": "D"})
print("tracked count equals Table 2 in", (trk.set_index("session").n_tracked == t2.n_tracked.reindex(trk.session).values).sum(), "of", len(trk))
tracked_sess = trk[trk.n_tracked > 0]
per_mouse = tracked_sess.groupby("subject_id").agg(group=("group", "first"), sessions=("session", "size"),
                                                   counts=("n_tracked", lambda x: sorted(set(x))),
                                                   same_ids=("ids", lambda x: len(set(x)) == 1),
                                                   min_rois=("n_rois", "min"), mean_rois=("n_rois", "mean"))
per_mouse["n_tracked"] = per_mouse.counts.str[0]
per_mouse["analysed"] = per_mouse.index.isin(analysed.mouseid)
per_mouse["tracked_in_sessions_1_3"] = [set(tracked_sess.loc[tracked_sess.subject_id == m, "session"]) <= set(analysed.sessid)
                                        for m in per_mouse.index]
print(per_mouse.to_string())
pm = per_mouse[per_mouse.analysed]
by_plane = pm.groupby("group").agg(tracked=("n_tracked", "sum"), max_a=("min_rois", "sum"), max_b=("mean_rois", "sum"))
by_plane["frac_a_%"] = 100 * by_plane.tracked / by_plane.max_a
by_plane["frac_b_%"] = 100 * by_plane.tracked / by_plane.max_b
print(by_plane.round(1).to_string())
frac_c = 100 * pm.n_tracked / pm.min_rois
print("(c) per mouse:", frac_c.round(1).to_dict())
for name, x in (("a", by_plane["frac_a_%"]), ("b", by_plane["frac_b_%"]), ("c", frac_c)):
    print(f"reading ({name}): {x.min():.1f} to {x.max():.1f} %, inside 28.8 to 82.5: {x.min() >= 28.8 and x.max() <= 82.5}")

**Result: reproduced.** Summed over the 11 analysed mice, the files hold 254 (L2/3-D), 261 (L2/3-S),
516 (L5-D) and 129 (L5-S) tracked ROIs, the paper's counts, and each mouse has the same tracking indices
in its three tracked sessions. Under reading (a) of the theoretical maximum the fractions run from 32.9
to 75.0%, inside the widened range; under (b) they run from 25.4 to 65.7% and under (c) from 25.2 to
80.5%, both outside it.

## fig8-example-matches

> In the example L2/3-D mouse, taking the union of matches over all session orderings and removing four
> conflicting matches gave 136 final matches; in the example L5-S mouse, removing two conflicts gave 47.

The examples are mice 413663 (L2/3-D) and 411771 (L5-S), as the claims file gives them from the paper's
code. The conflict counts cannot be tested from the files.

In [ ]:
for mouse in ("413663", "411771"):
    r = per_mouse.loc[mouse]
    print(mouse, r.group, "tracked ROIs:", r.n_tracked, "| sessions:", r.sessions, "| same indices:", r.same_ids)

**Result: reproduced.** The files give 136 tracked ROIs for 413663 and 47 for 411771, the paper's final
match counts. The conflicts removed (4 and 2) are not in the files and are not tested.

## fig8-ordering-variation

> The number of matched ROIs depends on the order in which sessions are aligned, and this variation is
> far smaller for somata than for dendrites.

Tested in the second notebook, `roi_matching_orderings.ipynb`, which reruns the matching with the public
n-way matching package over the six session orderings (it takes about twelve minutes).

## stimulus-durations

> Each imaging session lasts 70 minutes, with 34 minutes of Gabor sequences and 17 minutes of visual
> flow in each direction, each stimulus period flanked by 30 s of grey screen.

From the trial table of each analysed session: each stimulus period is its first row start to its last
row stop, per Gabor and per flow direction; grey periods are the grey-screen rows.

In [ ]:
dur = []
for s in sorted(ANALYSED):
    t = trials[s]
    row = {"session": s}
    for name, sel in (("gabors", t.stimulus_type == "gabors"),
                      ("flow_left", t.stimulus_template_name == "visflow_left"),
                      ("flow_right", t.stimulus_template_name == "visflow_right")):
        row[name + "_min"] = (t.stop_time[sel].max() - t.start_time[sel].min()) / 60
    g = t[t.stimulus_type == "grayscreen"]
    row["grey_n"] = len(g)
    row["grey_min_s"], row["grey_max_s"] = (g.stop_time - g.start_time).min(), (g.stop_time - g.start_time).max()
    order = t.stimulus_template_name.values
    row["order"] = " ".join(pd.Series(order)[pd.Series(order) != pd.Series(order).shift()].str.replace("visflow_", "").str.replace("grayscreen", "grey"))
    dur.append(row)
dur = pd.DataFrame(dur).set_index("session")
print(dur.drop(columns="order").describe().loc[["min", "50%", "max"]].round(3))
print(dur.order.value_counts().to_string())
ok = (dur.gabors_min.sub(34).abs() <= 3.4) & dur.flow_left_min.sub(17).abs().le(1.7) & dur.flow_right_min.sub(17).abs().le(1.7) \
     & dur.grey_min_s.sub(30).abs().le(3) & dur.grey_max_s.sub(30).abs().le(3)
print("sessions within tolerance:", ok.sum(), "of", len(ok))

**Result: reproduced.** In all 33 sessions the Gabor period lasts 34.03 to 34.13 minutes, each flow
direction 17.01 to 17.04 minutes, and each of the four grey periods 29.98 to 30.04 s, one before and
after each stimulus period.

## inconsistent-event-fraction

> Inconsistent sequences are about 7% of Gabor sequences and 5% of visual flow time, in about 30 blocks
> per session, each 30 to 90 s of consistent stimulus followed by 3 to 6 s (Gabor) or 2 to 4 s (visual
> flow) of inconsistent stimulus.

Gabor fraction: sequences with a U image over all sequences (D or U). Flow fraction: time of rows with
reversed squares over all flow time. A block is a run of consistent rows followed by a run of
inconsistent rows, within the Gabor period and within each flow direction (the stimulus code samples
them so). Durations are from row start and stop times. Readings: (a) mean of per-session fractions, (b)
pooled; blocks (i) per stimulus, (ii) per session.

In [ ]:
def runs(t):
    # consistent and inconsistent run durations, in order, for one stimulus period (rows in time order)
    u = t.unexpected.values == 1
    edges = np.flatnonzero(np.diff(u.astype(int))) + 1
    bounds = np.r_[0, edges, len(u)]
    out = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        out.append((bool(u[a]), t.stop_time.values[b - 1] - t.start_time.values[a]))
    return out

inc = []
cons_d, inc_g, inc_f = [], [], []
for s in sorted(ANALYSED):
    t = trials[s]
    gab = t[t.stimulus_type == "gabors"]
    n_d, n_u = (gab.gabor_frame == "D").sum(), (gab.gabor_frame == "U").sum()
    vf = t[t.stimulus_type == "visflow"]
    vf_dur = vf.stop_time - vf.start_time
    blocks = {}
    for name, sel in (("gabors", gab), ("visflow_left", vf[vf.stimulus_template_name == "visflow_left"]),
                      ("visflow_right", vf[vf.stimulus_template_name == "visflow_right"])):
        r = runs(sel)
        # a block: a consistent run followed by an inconsistent one; a trailing consistent run is not a block
        nb = 0
        for (u1, d1), (u2, d2) in zip(r[:-1], r[1:]):
            if not u1 and u2:
                nb += 1
                cons_d.append((name, d1))
                (inc_g if name == "gabors" else inc_f).append(d2)
        blocks[name] = nb
        blocks[name + "_trailing_s"] = r[-1][1] if not r[-1][0] else 0.0
        blocks[name + "_first_inconsistent"] = r[0][0]
    inc.append({"session": s, "gabor_U": n_u, "gabor_seq": n_d + n_u, "gabor_frac_%": 100 * n_u / (n_d + n_u),
                "flow_incons_s": vf_dur[vf.unexpected == 1].sum(), "flow_s": vf_dur.sum(),
                "flow_frac_%": 100 * vf_dur[vf.unexpected == 1].sum() / vf_dur.sum(),
                "blocks_gabor": blocks["gabors"], "blocks_flow": blocks["visflow_left"] + blocks["visflow_right"],
                "trailing_gabor_s": blocks["gabors_trailing_s"],
                "starts_inconsistent": any(blocks[k + "_first_inconsistent"] for k in ("gabors", "visflow_left", "visflow_right"))})
inc = pd.DataFrame(inc).set_index("session")
inc["blocks_session"] = inc.blocks_gabor + inc.blocks_flow
print(inc.describe().loc[["mean", "min", "max"]].round(2).to_string())
print("(a) mean of per-session fractions: Gabor %.2f%%, flow %.2f%%" % (inc["gabor_frac_%"].mean(), inc["flow_frac_%"].mean()))
print("(b) pooled: Gabor %.2f%%, flow %.2f%%" % (100 * inc.gabor_U.sum() / inc.gabor_seq.sum(), 100 * inc.flow_incons_s.sum() / inc.flow_s.sum()))
print("(i) per stimulus, mean blocks: Gabor %.1f, flow %.1f; (ii) per session %.1f" % (inc.blocks_gabor.mean(), inc.blocks_flow.mean(), inc.blocks_session.mean()))
cd = pd.DataFrame(cons_d, columns=["stim", "s"])
print("consistent run before each inconsistent run, s:", cd.groupby(cd.stim.str[:7]).s.agg(["min", "max", "size"]).round(2).to_dict("index"))
print("inconsistent runs, s: Gabor %.2f to %.2f (n %d); flow %.2f to %.2f (n %d)" % (min(inc_g), max(inc_g), len(inc_g), min(inc_f), max(inc_f), len(inc_f)))
print("Gabor inconsistent run lengths (s, rounded to 0.1):", pd.Series(np.round(inc_g, 1)).value_counts().sort_index().to_dict())
print("flow inconsistent run lengths (s, rounded to 0.1):", pd.Series(np.round(inc_f, 1)).value_counts().sort_index().to_dict())
out_c = cd[(cd.s < 30) | (cd.s > 90)]
print("consistent runs outside 30-90 s:", len(out_c), "of", len(cd), "| furthest outside by",
      round(max((30 - out_c.s).max(), (out_c.s - 90).max()), 3) if len(out_c) else 0, "s")
out_g = [x for x in inc_g if not 3 <= x <= 6]
out_f = [x for x in inc_f if not 2 <= x <= 4]
print("inconsistent runs outside the stated range: Gabor", len(out_g), "of", len(inc_g), "(largest", round(max(out_g, default=0), 3), "s)",
      "| flow", len(out_f), "of", len(inc_f), "(range", round(min(out_f, default=0), 3), "to", round(max(out_f, default=0), 3), "s)")
# The Methods sample durations on a grid (multiples of 1.5 s for Gabor, 1 s for flow); on that grid:
grid = np.where(cd.stim == "gabors", 1.5, 1.0)
cd_grid = np.round(cd.s / grid) * grid
print("on the sampling grid: consistent runs outside 30-90 s:", int(((cd_grid < 30) | (cd_grid > 90)).sum()),
      "| Gabor inconsistent outside 3-6 s:", int(sum(not 3 <= np.round(x / 1.5) * 1.5 <= 6 for x in inc_g)),
      "| flow inconsistent outside 2-4 s:", int(sum(not 2 <= np.round(x) <= 4 for x in inc_f)))
print("largest distance of a measured run from its grid value, s:",
      round(max(np.abs(cd.s - cd_grid).max(), max(abs(x - np.round(x / 1.5) * 1.5) for x in inc_g), max(abs(x - np.round(x)) for x in inc_f)), 3))
print("per session: Gabor fraction", inc["gabor_frac_%"].min().round(2), "to", inc["gabor_frac_%"].max().round(2),
      "%; flow fraction", inc["flow_frac_%"].min().round(2), "to", inc["flow_frac_%"].max().round(2), "%")

**Result: reproduced.** Inconsistent sequences are 6.93% of Gabor sequences and 4.65% of flow time
(readings a and b agree; per session 6.03 to 7.94% and 3.92 to 5.49%). There are on average 31.5 Gabor
blocks and 32.2 flow blocks per session (reading i, within 30 +- 3; reading ii, 63.7 per session, is
not). Measured from row times, 49 of 2101 consistent runs exceed 90 s by up to 0.22 s, and 377 of 1039
Gabor and 347 of 1062 flow inconsistent runs fall outside 3 to 6 s and 2 to 4 s by at most 0.06 s; on
the sampling grid of the Methods (1.5 s and 1 s steps) every run is within the stated ranges.

## fig5-stimulus-responses

> Somatic and dendritic ROIs respond to both the Gabor and the visual flow stimuli, many with increased
> fluorescence at stimulus onset, and their responses differ between consistent and inconsistent stimuli.

Session 1 of the 11 analysed mice, ROIs pooled per plane. Gabor: per ROI, mean dF/F over 0 to 0.6 s
after D against after U. Flow, each direction: mean over 0 to 2 s after against 2 s before inconsistent
flow onset. ROI selection (a) all ROIs, (b) the Fig. 5 legend's criteria. Tests (i) two-sided Wilcoxon
signed-rank across ROIs, p < 0.05, (ii) |mean difference| > 2 SEM. Unscaled dF/F as the claims file
says; the code's scaled traces are shown as a variant.

In [ ]:
def two_tests(x, y):
    dlt = np.asarray(y) - np.asarray(x)
    n = len(dlt)
    if n < 2:
        return n, np.nan, np.nan, np.nan, False, False
    p = scipy.stats.wilcoxon(dlt).pvalue if np.any(dlt != 0) else 1.0
    mean, sem = dlt.mean(), dlt.std(ddof=1) / np.sqrt(n)
    return n, mean, sem, p, p < 0.05, abs(mean) > 2 * sem

rows5 = []
for key in ("fig5", "fig5_scaled"):
    for s in sorted(SESS1):
        m = meta[s]
        grp = md_.set_index("session").loc[s]
        for stim, v in roi[s][key].items():
            for sel_name in ("a", "b"):
                keep = np.ones(len(v["consistent"]), bool) if sel_name == "a" else v["selected"]
                rows5.append({"traces": key, "plane": grp.line + "-" + ("S" if grp.plane == "soma" else "D"),
                              "mouse": m["subject_id"], "stim": stim, "selection": sel_name,
                              "consistent": v["consistent"][keep], "inconsistent": v["inconsistent"][keep]})
r5 = pd.DataFrame(rows5)
res5 = []
for (traces, plane, stim, sel), g in r5.groupby(["traces", "plane", "stim", "selection"]):
    x, y = np.concatenate(g.consistent.values), np.concatenate(g.inconsistent.values)
    n, mean, sem, p, wil, sem2 = two_tests(x, y)
    res5.append({"traces": traces, "plane": plane, "stim": stim, "selection": sel, "n_rois": n,
                 "mean_incons_minus_cons": mean, "sem": sem, "wilcoxon_p": p, "i_wilcoxon": wil, "ii_2sem": sem2})
res5 = pd.DataFrame(res5)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(res5[res5.traces == "fig5"].drop(columns="traces").to_string(index=False))
summary5 = res5.groupby(["traces", "selection", "plane"]).agg(any_i=("i_wilcoxon", "any"), any_ii=("ii_2sem", "any"))
print(summary5.to_string())
for traces in ("fig5", "fig5_scaled"):
    s5 = summary5.loc[traces]
    print(traces, "| every reading pair has a difference in all four planes:", bool(s5.all().all()))

In [ ]:
# By mouse: Gabor U minus D (all ROIs, unscaled), and the per-mouse Wilcoxon p.
bym = []
for _, g in r5[(r5.traces == "fig5") & (r5.selection == "a")].iterrows():
    n, mean, sem, p, wil, sem2 = two_tests(g.consistent, g.inconsistent)
    bym.append({"plane": g.plane, "mouse": g.mouse, "stim": g.stim, "n": n, "mean": mean, "sem": sem, "p": p})
with pd.option_context("display.float_format", "{:.3g}".format):
    print(pd.DataFrame(bym).pivot_table(index=["plane", "mouse"], columns="stim", values=["mean", "p"]).to_string())

**Result: reproduced.** Under all four reading pairs (selection a or b, test i or ii), each of the four
planes has at least one stimulus with a difference between consistent and inconsistent responses, and
the same holds with the code's scaled traces. With all ROIs (a), the Gabor difference is significant in
L2/3-D and L5-S but not in L2/3-S or L5-D, where only the flow differs. Under selection (b) the flow
selection is made on the after-minus-before difference itself, so its test is not independent of the
selection.

## fig6-behaviour

> Running velocity and pupil diameter were recorded in every analysed session, for the 11 mice whose
> data are shown aggregated in Fig. 6b.

Each analysed session must have a running velocity trace (cm/s) and a pupil diameter trace (mm) that
span the session, here the stimulus period from the first to the last trial-table row. Running is
median-filtered (5 frames) and single-frame changes of at least 50 cm/s are dropped; pupil frames with a
single-frame change of at least 0.05 mm are dropped (Methods).

In [ ]:
cov = []
for s in sorted(ANALYSED):
    m, t, b = meta[s], trials[s], beh[s]
    t0, t1 = t.start_time.min(), t.stop_time.max()
    cov.append({"session": s, "mouse": m["subject_id"], "run_unit": m["run_unit"], "pupil_unit": m["pupil_unit"],
                "run_spans": m["run_first"] <= t0 + 1 and m["run_last"] >= t1 - 1,
                "pupil_spans": m["pupil_first"] <= t0 + 1 and m["pupil_last"] >= t1 - 1,
                "run_n": m["run_n"], "run_kept_%": 100 * len(b["run"]) / m["run_n"],
                "pupil_nan_%": 100 * b["pupil_nan"] / m["pupil_n"], "pupil_kept_%": 100 * len(b["pupil"]) / m["pupil_n"],
                "run_median": np.median(b["run"]), "pupil_median": np.median(b["pupil"])})
cov = pd.DataFrame(cov)
print(cov.drop(columns=["session"]).describe().loc[["min", "50%", "max"]].round(3).to_string())
print("units:", cov.run_unit.unique(), cov.pupil_unit.unique())
ok6 = cov[cov.run_spans & cov.pupil_spans & (cov.run_unit == "cm/s") & (cov.pupil_unit == "mm")]
print("sessions with both traces spanning the session:", len(ok6), "of", len(cov), "| mice:", ok6.mouse.nunique())
print(cov.groupby("mouse")[["pupil_nan_%", "run_median", "pupil_median"]].median().round(3).to_string())

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(11, 5), sharey="row")
for col, sn in enumerate((1, 2, 3)):
    ss = analysed.loc[analysed.sess_n == sn, "sessid"]
    run = [beh[s]["run"] for s in ss]
    pup = [beh[s]["pupil"] for s in ss]
    ax[0, col].hist(run, bins=np.linspace(-20, 80, 51), stacked=True, color=plt.cm.Greys(np.linspace(0.3, 0.9, len(run))))
    ax[1, col].hist(pup, bins=np.linspace(0, 1.5, 51), stacked=True, color=plt.cm.Greys(np.linspace(0.3, 0.9, len(pup))))
    ax[0, col].set_title(f"session {sn}")
    out_r = sum(int(((x < -20) | (x > 80)).sum()) for x in run)
    out_p = sum(int(((x < 0) | (x > 1.5)).sum()) for x in pup)
    print(f"session {sn}: running {min(x.min() for x in run):.1f} to {max(x.max() for x in run):.1f} cm/s, {out_r} values off the axes; "
          f"pupil {min(x.min() for x in pup):.3f} to {max(x.max() for x in pup):.3f} mm, {out_p} off the axes")
ax[0, 0].set_ylabel("running frames"); ax[1, 0].set_ylabel("pupil frames")
for a in ax[0]: a.set_xlabel("running velocity (cm/s)")
for a in ax[1]: a.set_xlabel("pupil diameter (mm)")
fig.tight_layout()

**Result: reproduced.** All 33 analysed sessions, from 11 mice, have a running velocity trace in cm/s and
a pupil diameter trace in mm that span the stimulus period; 0.06 to 2.3% of pupil frames are NaN.

## fig7a-snr-high-stable

> ROI SNR is relatively high, above 1, in every plane and session, and stable over the three days.

SNR per ROI as in the paper's code, ROIs pooled across mice within each line, plane and session number
(12 groups). "Above 1": (a) group median > 1, (b) group 25th percentile > 1. "Stable": (i) within each
plane, each session's median lies inside every other session's 25th to 75th percentile box; (ii) each
session's median within 10% of the mean of the three medians.

In [ ]:
def per_roi(key):
    rows = []
    for s, v in roi.items():
        m = md_.set_index("session").loc[s]
        rows.append(pd.DataFrame({"value": v[key], "session": s, "mouse": m.subject_id, "sess_n": m.sess_n,
                                  "plane": m.line + "-" + ("S" if m.plane == "soma" else "D")}))
    return pd.concat(rows, ignore_index=True)

def group_stats(df):
    g = df.groupby(["plane", "sess_n"]).value
    return pd.DataFrame({"n": g.size(), "q25": g.quantile(0.25), "median": g.median(), "q75": g.quantile(0.75),
                         "q05": g.quantile(0.05), "q95": g.quantile(0.95)})

def stability(st):
    out = {}
    for plane, g in st.groupby(level="plane"):
        med, q25, q75 = g["median"].values, g.q25.values, g.q75.values
        inside = all(q25[j] <= med[i] <= q75[j] for i in range(3) for j in range(3) if i != j)
        within = bool(np.all(np.abs(med - med.mean()) <= 0.1 * med.mean()))
        out[plane] = {"i_inside_boxes": inside, "ii_within_10%": within,
                      "max_dev_from_mean_%": 100 * np.max(np.abs(med - med.mean())) / med.mean()}
    return pd.DataFrame(out).T

snr = per_roi("snr")
st = group_stats(snr)
print(st.round(3).to_string())
print("(a) every group median > 1:", bool((st["median"] > 1).all()), "| (b) every group 25th percentile > 1:", bool((st.q25 > 1).all()),
      "| smallest 25th percentile:", st.q25.min().round(3))
print(stability(st).to_string())
print("ROIs with SNR <= 1:", int((snr.value <= 1).sum()), "of", len(snr))
print(snr.groupby(["plane", "mouse", "sess_n"]).value.median().unstack().round(2).to_string())

**Result: reproduced.** Every group's median and 25th percentile of SNR are above 1 (smallest 25th
percentile 2.32; no ROI has SNR at or below 1), and in all four planes each session's median lies within
every other session's interquartile box and within 4.8% of the mean of the three medians, so every
reading pair holds.

## fig7b-signal-stable

> The mean dF/F signal of the ROIs is stable over days.

Signal is the mean of an ROI's dF/F values above its noise threshold. Readings (i) and (ii) as for SNR.

In [ ]:
sig = per_roi("signal")
st_b = group_stats(sig)
print(st_b.round(4).to_string())
print(stability(st_b).to_string())
print(sig.groupby(["plane", "mouse", "sess_n"]).value.median().unstack().round(3).to_string())

**Result: reproduced.** Both stability readings hold in all four planes: each session's median signal
lies inside the other sessions' boxes and within 9.4% (L5-D, the largest) of the mean of the three
medians.

## fig7c-correlations-similar

> The distributions of pairwise ROI correlations are very similar for apical dendrite ROIs and for
> somatic ROIs, so the dendrites are unlikely to be heavily over-segmented.

Pearson correlations over all ROI pairs of each session (4-sample rolling mean), pooled across the mice
of each line, plane and session. Six comparisons (line x session), dendritic against somatic: (i) the
dendritic median inside the somatic 25th to 75th percentile range; (ii) the dendritic 99th percentile
at most 10% above the somatic 99th percentile.

In [ ]:
corr_rows = []
pooled = {}
for (line, plane, sn), g in md_[md_.analysed].groupby(["line", "plane", "sess_n"]):
    c = np.concatenate([roi[s]["corr"] for s in g.session])
    pooled[(line, plane, sn)] = c
    q = np.quantile(c, [0.25, 0.5, 0.75, 0.99])
    corr_rows.append({"line": line, "plane": plane, "sess_n": sn, "pairs": c.size, "q25": q[0], "median": q[1], "q75": q[2],
                      "q99": q[3], "frac_above_0.8_%": 100 * np.mean(c > 0.8)})
cr = pd.DataFrame(corr_rows).set_index(["line", "sess_n", "plane"])
with pd.option_context("display.float_format", "{:.4g}".format):
    print(cr.to_string())
comp = []
for (line, sn), g in cr.groupby(level=["line", "sess_n"]):
    d, s = g.xs("dend", level="plane").iloc[0], g.xs("soma", level="plane").iloc[0]
    comp.append({"line": line, "sess_n": sn, "dend_median": d["median"], "soma_q25": s.q25, "soma_q75": s.q75,
                 "i": s.q25 <= d["median"] <= s.q75, "dend_q99": d.q99, "soma_q99": s.q99, "ii": d.q99 <= 1.1 * s.q99,
                 "ks_stat": scipy.stats.ks_2samp(pooled[(line, "dend", sn)], pooled[(line, "soma", sn)]).statistic})
comp = pd.DataFrame(comp)
with pd.option_context("display.float_format", "{:.4g}".format):
    print(comp.to_string(index=False))
print("(i) holds in", comp.i.sum(), "of 6; (ii) holds in", comp.ii.sum(), "of 6")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
bins = np.linspace(-0.3, 1, 66)
for k, line in enumerate(("L2/3", "L5")):
    for plane, ls in (("dend", "-"), ("soma", "--")):
        for sn in (1, 2, 3):
            c = pooled[(line, plane, sn)]
            ax[k].hist(c, bins=bins, histtype="step", density=True, ls=ls, label=f"{plane} s{sn}")
            print(line, plane, sn, "range", c.min().round(3), "to", c.max().round(3), "| pairs off the axes:", int(((c < -0.3) | (c > 1)).sum()))
    ax[k].set_yscale("log"); ax[k].set_title(line); ax[k].set_xlabel("pairwise correlation")
ax[0].legend(fontsize=7); ax[0].set_ylabel("density")
fig.tight_layout()

**Result: blocked (methods underspecified).** Reading (ii) holds in all six comparisons: the dendritic
99th percentile is below the somatic one in each (0.081 to 0.119 against 0.175 to 0.207). Reading (i)
holds in 5 of 6: in L5 session 3 the dendritic median, 0.0186, is just below the somatic 25th
percentile, 0.0190. The criterion asks both readings to hold in all six for a reproduction and blocks
the claim when they disagree.

## high-correlation-pairs

> About 0.01% of possible pairs of L2/3 dendritic ROIs have correlations above 0.8.

Readings for the denominator: (a) pairs pooled over the L2/3-D sessions 1 to 3 of the analysed mice; (b)
mean of per-session fractions; (c) session 1 only (pooled over its mice).

In [ ]:
l23d = md_[md_.analysed & (md_.line == "L2/3") & (md_.plane == "dend")]
per_s = pd.DataFrame({s: {"mouse": meta[s]["subject_id"], "sess_n": l23d.set_index("session").loc[s, "sess_n"],
                          "pairs": roi[s]["corr"].size, "above": int(np.sum(roi[s]["corr"] > 0.8))} for s in l23d.session}).T
per_s["frac_%"] = 100 * per_s.above.astype(float) / per_s.pairs.astype(float)
print(per_s.to_string())
a = 100 * per_s.above.sum() / per_s.pairs.sum()
b = per_s["frac_%"].mean()
s1 = per_s[per_s.sess_n == 1]
c = 100 * s1.above.sum() / s1.pairs.sum()
print(f"(a) pooled {a:.4f}%  (b) mean of sessions {b:.4f}%  (c) session 1 {c:.4f}%  | tolerance 0.005 to 0.015%")
other = cr["frac_above_0.8_%"].groupby(level=["line", "plane"]).mean()
print("mean over sessions of the pooled fraction above 0.8 (%), other planes:", other.round(4).to_dict())

**Result: not reproduced.** 0.0178% of L2/3-D pairs have correlations above 0.8 when pooled over the
six sessions (reading a), 0.0175% as a mean of sessions (b), and 0.0211% in session 1 (c), all above the
0.005 to 0.015% the claim allows. Per session the fraction ranges from 0.0018 to 0.046%, and most of the
high pairs come from sessions 1 and 2 of 413663.

## fig9-onset-offset-transients

> Mean dF/F traces show distinct transients aligned with the stimulus onsets and offsets (grey screen to
> stimulus and back), in every plane and session.

Per ROI, the mean over events of (mean dF/F 0 to 2 s after - mean over 2 s before), for onsets (grey to
stimulus) and offsets (stimulus to grey), both stimuli. Readings (i) |mean across ROIs| > 2 SEM, (ii)
two-sided Wilcoxon signed-rank across ROIs, p < 0.05. Variants: Gabor onsets and offsets only, and the
code's scaled traces.

In [ ]:
res9 = []
for key in ("onset_diff", "offset_diff", "onset_diff_gabor", "offset_diff_gabor", "onset_diff_scaled", "offset_diff_scaled"):
    for (line, plane, sn), g in md_[md_.analysed].groupby(["line", "plane", "sess_n"]):
        dlt = np.concatenate([roi[s][key] for s in g.session])
        n, mean, sem, p, wil, sem2 = two_tests(np.zeros_like(dlt), dlt)
        res9.append({"measure": key, "group": f"{line}-{'S' if plane == 'soma' else 'D'}", "sess_n": sn, "n_rois": n,
                     "mean": mean, "sem": sem, "wilcoxon_p": p, "i_2sem": sem2, "ii_wilcoxon": wil})
res9 = pd.DataFrame(res9)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(res9[res9.measure.isin(["onset_diff", "offset_diff"])].to_string(index=False))
print(res9.groupby("measure")[["i_2sem", "ii_wilcoxon"]].sum().astype(int).astype(str).add(" of 12").to_string())

In [ ]:
# By mouse (unscaled, both stimuli): mean difference and Wilcoxon p.
bym9 = []
for key in ("onset_diff", "offset_diff"):
    for s in sorted(ANALYSED):
        m = md_.set_index("session").loc[s]
        n, mean, sem, p, wil, sem2 = two_tests(np.zeros(len(roi[s][key])), roi[s][key])
        bym9.append({"measure": key, "plane": m.line + "-" + m.plane, "mouse": m.subject_id, "sess_n": m.sess_n, "mean": mean, "p": p})
bym9 = pd.DataFrame(bym9)
with pd.option_context("display.float_format", "{:.2g}".format):
    print(bym9.pivot_table(index=["plane", "mouse"], columns=["measure", "sess_n"], values="mean").to_string())
print("mouse-sessions with p < 0.05: onset", int((bym9[bym9.measure == "onset_diff"].p < 0.05).sum()),
      "| offset", int((bym9[bym9.measure == "offset_diff"].p < 0.05).sum()), "of", len(ANALYSED))

In [ ]:
# Mean traces (+-SEM across ROIs) around onsets and offsets, per group, unscaled, both stimuli.
tt = np.arange(-ca.PRE, ca.POST) / ca.FPS
fig, ax = plt.subplots(2, 4, figsize=(13, 5), sharex=True)
for k, (line, plane) in enumerate([("L2/3", "dend"), ("L2/3", "soma"), ("L5", "dend"), ("L5", "soma")]):
    for r, key in enumerate(("onset_trace", "offset_trace")):
        for sn in (1, 2, 3):
            ss = md_[md_.analysed & (md_.line == line) & (md_.plane == plane) & (md_.sess_n == sn)].session
            tr_ = np.concatenate([roi[s][key] for s in ss])
            mu, se = tr_.mean(0), tr_.std(0, ddof=1) / np.sqrt(len(tr_))
            ax[r, k].plot(tt, mu, label=f"session {sn}"); ax[r, k].fill_between(tt, mu - se, mu + se, alpha=0.3)
            print(line, plane, key, sn, "mean trace range", mu.min().round(4), "to", mu.max().round(4))
        ax[r, k].axvline(0, ls="--", c="k"); ax[r, k].set_title(f"{line} {plane} {key.split('_')[0]}", fontsize=9)
ax[0, 0].legend(fontsize=7)
for a in ax[1]: a.set_xlabel("time from event (s)")
for a in ax[:, 0]: a.set_ylabel("dF/F")
fig.tight_layout()

**Result: reproduced.** Both readings find a transient at onset and at offset in all 12 groups. The
mean dF/F rises after stimulus onsets (mean differences 0.049 to 0.103 dF/F) and is lower after offsets
than before them (-0.013 to -0.035), with a brief peak after the offset in the mean traces. With Gabor
events only, onsets give a difference in 12 of 12 groups and offsets in 7 (i) and 9 (ii) of 12.

## Summary

| Claim | Outcome |
|---|---|
| dataset-holds-paper-sessions | not reproduced: 000336 holds none of the paper's sessions |
| table1-subjects | reproduced (on 000037) |
| table2-sessions | reproduced (on 000037) |
| session-schedule | reproduced (on 000037), under readings (a) and (ii) |
| imaging-planes | partially reproduced (on 000037): 433414 is at 60 um in the files, 75 um in Table 2 |
| roi-counts-per-session | reproduced (on 000037) |
| tracked-rois-per-plane | reproduced (on 000037), fractions under reading (a) |
| fig8-example-matches | reproduced (on 000037) |
| fig8-ordering-variation | reproduced (on 000037), in `roi_matching_orderings.ipynb`, resting on assumptions |
| stimulus-durations | reproduced (on 000037) |
| inconsistent-event-fraction | reproduced (on 000037), blocks under reading (i) |
| fig5-stimulus-responses | reproduced (on 000037) |
| fig6-behaviour | reproduced (on 000037) |
| fig7a-snr-high-stable | reproduced (on 000037) |
| fig7b-signal-stable | reproduced (on 000037) |
| fig7c-correlations-similar | blocked, methods underspecified: the criterion's two readings disagree in one of six comparisons |
| high-correlation-pairs | not reproduced (on 000037): about 0.018% of pairs, against 0.01 +- 0.005% |
| fig9-onset-offset-transients | reproduced (on 000037) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the target dandiset, 000336, does not hold the paper's data,
so every finding was tested on DANDI:000037 (version 0.230426.0054). Session numbers and QC come from the
paper's code table, as the files carry neither. Onset, offset and Fig. 5 measures use unscaled dF/F, as
the claims file says, with the code's scaled traces as a variant. The Fig. 5 test and ROI selection, the
stability and similarity readings and the theoretical maximum of tracked ROIs are the claims file's
readings, since the paper gives no tests. The ordering analysis uses the public matching package, not
the paper's modified one, with the overlap constraint read as intersection over union.

What was found about the datasets:

- DANDI:000336 names the paper as a related resource but holds a different OpenScope project
  (mesoscope imaging of 16 other mice, 2022 to 2023); the paper's data are in DANDI:000037.
- In DANDI:000037 the stored `session_start_time` carries a Pacific offset (-07:00 or -08:00) on a clock
  time that is the UTC start time, so the stored times are 7 to 8 hours late; read as written, 4 of 50
  sessions fall on a different date from Table 2.
- The three sessions of 433414 give a depth of 60 um, where Table 2 and the paper's code table give 75 um.
- The files carry no QC status or session number, which Table 2 and the paper's code give.
- Session 759872185 (411400, failed QC) has 61134 pupil samples for 127588 imaging frames.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")